### MFCC

In [ ]:
# read features file
import pandas as pd

df8918 = pd.read_pickle('*******.pkl')
df8918

In [ ]:
df8918['event_clinical_yesno'].value_counts()

In [ ]:
df8918 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918

In [ ]:
import os
import librosa


from spafe.features.mfcc import mfcc
import numpy as np
from scipy.io.wavfile import read
from spafe.utils.preprocessing import SlidingWindow

def mfcc_features_extractor(sig, fs):
    #fs, sig = read(file)
    mfccs  = mfcc(sig,
              fs=fs,
              pre_emph=1,
              pre_emph_coeff=0.97,
              window=SlidingWindow(0.025, 0.005, "hamming"),
              nfilts=80,
              nfft=2048,
              low_freq=0,
              high_freq=fs/2,
              #normalize="mvn",
              num_ceps=80,
             )
    mfccs_scaled_features = np.mean(mfccs,axis=0)
    
    return mfccs_scaled_features



root_dir = "********" 

result_df = pd.DataFrame()

#iterate though ROWS IN meta_id_speech
for index, row in df8918.iterrows():
    name = row['filename'] 
    sid = row['sid']
    path = f'uid0000{sid}/{name}' 
    audio_path = os.path.join(root_dir, path)
    if os.path.exists(audio_path):
        # Read audio files
        audio, sr = librosa.load(audio_path)
        
        # Feature extraction
        features = mfcc_features_extractor(audio, sr)
        
        #Convert features to DataFrame
        column_names = [f"mfcc{i}" for i in range(80)] 
        features_df = pd.DataFrame([features], columns=column_names) 
        # Add sid column
        features_df['sid'] = sid
        #Add filename column
        features_df['filename'] = name
        
        #Merge into the result DataFrame
        result_df = pd.concat([result_df, features_df], ignore_index=True)



In [ ]:
result_df.to_pickle('***********.pkl') # save MFCCs features from converted recordings to a pickle file

### egemaps

In [ ]:
import os
import pandas as pd
import opensmile
import librosa


# Initialize eGeMAPSv01b extractor
smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv01b,
            feature_level=opensmile.FeatureLevel.Functionals,
        )

root_dir = "**********/fullset/tacticas_converted" 


result_df = pd.DataFrame()

#iterate though ROWS IN meta_id_speech
for index, row in df8918.iterrows():
    name = row['filename'].split('/')[-1]
    sid = row['sid']
    path = f'uid0000{sid}/{name}' 
    audio_path = os.path.join(root_dir, path)
    if os.path.exists(audio_path):
        #Read audio files
        audio, sr = librosa.load(audio_path)
        
        #Feature extraction
        features = smile.process_signal(audio, sr)
        
        # Convert features to DataFrame
        features_df = pd.DataFrame(features)
        # Add sid column
        features_df['sid'] = sid
        # Add filename column
        features_df['filename'] = name
        
        # Merge into the result DataFrame
        result_df = pd.concat([result_df, features_df], ignore_index=True)

In [ ]:
result_df.to_pickle('*******.pkl')# save eGeMAPS features from converted recordings to a pickle file

In [ ]:
# read features file
import pandas as pd
df8918_egemaps = pd.read_pickle('*******.pkl')
df8918_egemaps

In [ ]:
df8918_mfcc = pd.read_pickle('*******.pkl')
df8918_mfcc 

In [ ]:
df8918_egemaps['filename_sid'] = df8918_egemaps['sid'].astype(str) + '_' + df8918_egemaps['filename']
df8918_mfcc['filename_sid'] = df8918_mfcc['sid'].astype(str)+ '_' +  df8918_mfcc['filename'] 
df8918_mfcc

In [ ]:
# merge the two dataframes on 'sid', 'filename', 'filename_sid'
df_merged = pd.merge(df8918_egemaps, df8918_mfcc, on=['sid', 'filename', 'filename_sid'], how='inner')
df_merged

In [ ]:
df8918_label = pd.read_pickle('*******.pkl')
df8918_label

In [ ]:
# give the 'event_clinical_yesno' and 'Illness' and 'label' columns to df_merged based on 'sid' and 'filename' and 'filename_sid'
df_final = pd.merge(df_merged, df8918_label[['sid', 'filename', 'filename_sid', 'event_clinical_yesno', 'Illness', 'label']], on=['sid', 'filename' ,'filename_sid'], how='left')

df_final

In [ ]:
final_df_select1  = df_final[['shimmerLocaldB_sma3nz_amean', 'shimmerLocaldB_sma3nz_stddevNorm',
                                             'loudness_sma3_amean', 'loudness_sma3_stddevNorm', 
                                             'HNRdBACF_sma3nz_amean', 'HNRdBACF_sma3nz_stddevNorm',

                                             'loudnessPeaksPerSec',
                                             'MeanVoicedSegmentLengthSec', 'StddevVoicedSegmentLengthSec',
                                              'MeanUnvoicedSegmentLength', 'StddevUnvoicedSegmentLength',
                                               'VoicedSegmentsPerSec',

                                               'F0semitoneFrom27.5Hz_sma3nz_amean','F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
                                             'jitterLocal_sma3nz_amean', 'jitterLocal_sma3nz_stddevNorm', 
                                              'F1frequency_sma3nz_amean', 'F1frequency_sma3nz_stddevNorm',  
                                              'F2frequency_sma3nz_amean',  'F2frequency_sma3nz_stddevNorm', 
                                               'F3frequency_sma3nz_stddevNorm','F3amplitudeLogRelF0_sma3nz_amean',
                                               'F1bandwidth_sma3nz_amean', 'F1bandwidth_sma3nz_stddevNorm',
                                               
                                               'mfcc0', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7',
                                               'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14', 'mfcc15',
                                               'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20',
                                               'mfcc21', 'mfcc22', 'mfcc23', 'mfcc24', 'mfcc25', 'mfcc26', 'mfcc27', 'mfcc28',
                                               'mfcc29', 


                                             'sid','filename', 'filename_sid', 'event_clinical_yesno', 'Illness', 'label'
                                             
                                             ]]

In [ ]:
final_df_select1

In [ ]:
final_df_select1['event_clinical_yesno'].value_counts()

In [ ]:
final_df_select1['label'].value_counts()

In [ ]:
final_df_select1.to_pickle('*******.pkl')

### Task1 _ Converted SVM

In [ ]:
# 《COPDVD: Automated classification of chronic obstructive pulmonary disease on a new collected and evaluated voice dataset》
# SVM Parameter adjustment, manual parameter tuning. The parameters are from this article.

In [ ]:
# read pickle file
import pandas as pd
df8918_egemaps = pd.read_pickle('********.pkl')
df8918_egemaps

In [ ]:
df8918_egemaps = df8918_egemaps.dropna()
df8918_egemaps

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

df_copd_asthma_speech = df8918_egemaps.copy()

# Create speaker encoder
le_speaker = LabelEncoder()
df_copd_asthma_speech['speaker_id'] = le_speaker.fit_transform(df_copd_asthma_speech['sid'])
num_speakers_total = len(le_speaker.classes_)

# Segmentation features and labels
features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_labels = df_copd_asthma_speech['speaker_id'].values.astype(np.int64)


In [ ]:
# Step 1: Divide the dataset into a test set (20%), and use the remaining 80% for training and validation, dividing it according to the distribution of y.
X_temp, X_test, y_temp, y_emo_test, sid_temp, y_spk_test = train_test_split(
    features, emotion_labels, speaker_labels, test_size=0.2, stratify=emotion_labels, random_state=42
)   
# Step 2: Divide the validation set (25% of 80% = 20%) and training set (75% of 80% = 60%) from the remaining 80%, and divide them according to the distribution of y
X_train, X_val, y_emo_train, y_emo_val, y_spk_train, y_spk_val = train_test_split(
    X_temp, y_temp, sid_temp, test_size=0.2, stratify=y_temp, random_state=42
)   

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)



In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score

# ==========================================
# 1. Definition of Functions
# ==========================================

def get_positive_prob(model, X, pos_label=1):
    """Securely obtain the predicted probability of the positive class, avoiding errors caused by the class label order not being [0, 1]."""
    classes = list(model.classes_)
    if pos_label not in classes:
        raise ValueError(f"pos_label={pos_label} not found in model.classes_: {classes}")
    pos_idx = classes.index(pos_label)
    return model.predict_proba(X)[:, pos_idx]

def _safe_confusion_matrix(y_true, y_pred):
    """Force labels=[0, 1] to prevent ravel() from throwing an error when the prediction is all 0 or all 1."""
    return confusion_matrix(y_true, y_pred, labels=[0, 1])

# ==========================================
# 2. The select_threshold function you provided
# ==========================================

def select_threshold(labels, probs, min_sensitivity=0.75, min_specificity=0.75, pos_label=1):
    labels = np.asarray(labels)
    probs = np.asarray(probs, dtype=np.float64)

    # Filter out NaN/INF
    valid_prob = np.isfinite(probs)
    labels = labels[valid_prob]
    probs = probs[valid_prob]

    # Convert labels to 0/1 for binary classification
    y = (labels == pos_label).astype(np.int64)

    if len(y) == 0 or len(np.unique(y)) < 2:
        print("Warning: Validation set invalid or single-class. Using 0.5.")
        return 0.5

    _, _, thresholds = roc_curve(y, probs)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    thresholds = thresholds[np.isfinite(thresholds)]
    
    # Add boundaries and restrict to [0, 1]
    thresholds = np.unique(np.concatenate([thresholds, np.array([0.0, 1.0])]))
    thresholds = thresholds[(thresholds >= 0.0) & (thresholds <= 1.0)]

    f1_scores, sens_scores, spec_scores = [], [], []

    for thresh in thresholds:
        preds = (probs >= thresh).astype(np.int64)
        f1_scores.append(float(f1_score(y, preds, zero_division=0)))
        
        cm = _safe_confusion_matrix(y, preds)
        tn, fp, fn, tp = cm.ravel()
        sens_scores.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        spec_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    # Logic 1: Simultaneously satisfy Sensitivity and Spec constraints
    valid_indices = [i for i in range(len(thresholds)) 
                     if sens_scores[i] >= min_sensitivity and spec_scores[i] >= min_specificity]

    if valid_indices:
        best_idx = valid_indices[int(np.argmax([f1_scores[i] for i in valid_indices]))]
    else:
        # Logic 2: As a fallback, only satisfy Sensitivity
        valid_sens_indices = [i for i, s in enumerate(sens_scores) if s >= min_sensitivity]
        if valid_sens_indices:
            print("Warning: Fallback to Sensitivity-only constraint.")
            best_idx = valid_sens_indices[int(np.argmax([f1_scores[i] for i in valid_sens_indices]))]
        else:
            # Logic 3: Maximize global F1
            print("Warning: Fallback to global best F1 threshold.")
            best_idx = int(np.argmax(f1_scores))

    return float(thresholds[best_idx])


# ==========================================
# 3. Model Preparation and Threshold Selection
# ==========================================

# Your optimal parameters
best_params = {'C': 10, 'kernel': 'rbf', 'degree': None, 'class_weight': None}

# Clean up parameters with values ​​of None to prevent SVC initialization errors
clean_params = {k: v for k, v in best_params.items() if v is not None}

# Reinitialize the optimal model (if your previous loop's `best_model` is still in memory, you can use `best_model` directly)
best_model = SVC(probability=True, random_state=42, **clean_params)

# Train on the training set (assuming `X_train` and `y_emo_train` have been processed by StandardScaler)
best_model.fit(X_train, y_emo_train) 

# Obtain probabilities and select a threshold on the validation set
val_prob = get_positive_prob(best_model, X_val, pos_label=1)

best_threshold = select_threshold(
    labels=y_emo_val, 
    probs=val_prob, 
    min_sensitivity=0.75, 
    min_specificity=0.75,
    pos_label=1
)

print(f"-"*40)
print(f"Selected Threshold on Validation Set: {best_threshold:.4f}")
print(f"-"*40)


# ==========================================
# 4. Final Test Set Evaluation
# ==========================================

# Obtain test set probabilities
test_prob = get_positive_prob(best_model, X_test, pos_label=1)

# Ensure test set labels are in 0/1 format
y_test_bin = (np.asarray(y_emo_test) == 1).astype(int)

# Apply the threshold selected from the validation set
y_test_pred = (test_prob >= best_threshold).astype(int)

# Calculate metrics
test_auc = roc_auc_score(y_test_bin, test_prob)
cm = _safe_confusion_matrix(y_test_bin, y_test_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity_final = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_final = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy_final = accuracy_score(y_test_bin, y_test_pred)

# Output the final result
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {accuracy_final:.4f}")
print(f"Test Sensitivity (Recall): {sensitivity_final:.4f}")
print(f"Test Specificity: {specificity_final:.4f}")
print("Test Confusion Matrix:")
print(cm)
print(f"(Format: [[TN, FP],\n          [FN, TP]])")

In [ ]:
#print best model parameters
print(f"Best Model Parameters: {clean_params}")

In [ ]:
print(f"\nbest_threshold ({best_threshold:.50f}):")

In [ ]:
task1_converted = test_prob

### Task1 _ None_converted SVM

In [ ]:
# read pickle file
import pandas as pd

df8918 = pd.read_pickle('******.pkl')
df8918

In [ ]:
df8918['event_clinical_yesno'].value_counts()

In [ ]:
df8918 = df8918[df8918['event_clinical_yesno'].isin([0, 1])]
df8918

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import roc_curve, accuracy_score, roc_auc_score, confusion_matrix
import numpy as np

df_copd_asthma_speech = df8918.copy()

# Create speaker encoder
le_speaker = LabelEncoder()
df_copd_asthma_speech['speaker_id'] = le_speaker.fit_transform(df_copd_asthma_speech['sid'])
num_speakers_total = len(le_speaker.classes_)

# Segmentation features and labels
features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_labels = df_copd_asthma_speech['speaker_id'].values.astype(np.int64)


In [ ]:
# Step 1: Divide the dataset into a test set (20%), and use the remaining 80% for training and validation, dividing it according to the distribution of y.
X_temp, X_test, y_temp, y_emo_test, sid_temp, y_spk_test = train_test_split(
    features, emotion_labels, speaker_labels, test_size=0.2, stratify=emotion_labels, random_state=42
)   
# # Step 2: Divide the validation set (25% of 80% = 20%) and training set (75% of 80% = 60%) from the remaining 80%, and divide them according to the distribution of y
X_train, X_val, y_emo_train, y_emo_val, y_spk_train, y_spk_val = train_test_split(
    X_temp, y_temp, sid_temp, test_size=0.2, stratify=y_temp, random_state=42
)   

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score

# ==========================================
# 1. Definition of Functions
# ==========================================

def get_positive_prob(model, X, pos_label=1):
    """Securely obtain the predicted probability of the positive class, avoiding errors caused by the class label order not being [0, 1]."""
    classes = list(model.classes_)
    if pos_label not in classes:
        raise ValueError(f"pos_label={pos_label} not found in model.classes_: {classes}")
    pos_idx = classes.index(pos_label)
    return model.predict_proba(X)[:, pos_idx]

def _safe_confusion_matrix(y_true, y_pred):
    """Force labels=[0, 1] to prevent ravel() from throwing an error when the prediction is all 0 or all 1."""
    return confusion_matrix(y_true, y_pred, labels=[0, 1])

# ==========================================
# 2. The select_threshold function you provided
# ==========================================

def select_threshold(labels, probs, min_sensitivity=0.75, min_specificity=0.75, pos_label=1):
    labels = np.asarray(labels)
    probs = np.asarray(probs, dtype=np.float64)

    # Filter out NaN/INF
    valid_prob = np.isfinite(probs)
    labels = labels[valid_prob]
    probs = probs[valid_prob]

    # Convert labels to 0/1 for binary classification
    y = (labels == pos_label).astype(np.int64)

    if len(y) == 0 or len(np.unique(y)) < 2:
        print("Warning: Validation set invalid or single-class. Using 0.5.")
        return 0.5

    _, _, thresholds = roc_curve(y, probs)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    thresholds = thresholds[np.isfinite(thresholds)]
    
    # Add boundaries and restrict to [0, 1]
    thresholds = np.unique(np.concatenate([thresholds, np.array([0.0, 1.0])]))
    thresholds = thresholds[(thresholds >= 0.0) & (thresholds <= 1.0)]

    f1_scores, sens_scores, spec_scores = [], [], []

    for thresh in thresholds:
        preds = (probs >= thresh).astype(np.int64)
        f1_scores.append(float(f1_score(y, preds, zero_division=0)))
        
        cm = _safe_confusion_matrix(y, preds)
        tn, fp, fn, tp = cm.ravel()
        sens_scores.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        spec_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    # Logic 1: Simultaneously satisfy Sensitivity and Spec constraints
    valid_indices = [i for i in range(len(thresholds)) 
                     if sens_scores[i] >= min_sensitivity and spec_scores[i] >= min_specificity]

    if valid_indices:
        best_idx = valid_indices[int(np.argmax([f1_scores[i] for i in valid_indices]))]
    else:
        # Logic 2: As a fallback, only satisfy Sensitivity
        valid_sens_indices = [i for i, s in enumerate(sens_scores) if s >= min_sensitivity]
        if valid_sens_indices:
            print("Warning: Fallback to Sensitivity-only constraint.")
            best_idx = valid_sens_indices[int(np.argmax([f1_scores[i] for i in valid_sens_indices]))]
        else:
            # Logic 3: Maximize global F1
            print("Warning: Fallback to global best F1 threshold.")
            best_idx = int(np.argmax(f1_scores))

    return float(thresholds[best_idx])


# ==========================================
# 3. Model Preparation and Threshold Selection
# ==========================================

# Your optimal parameters
best_params = {'C': 10, 'kernel': 'rbf', 'degree': None, 'class_weight': None}

# Clean up parameters with values ​​of None to prevent SVC initialization errors
clean_params = {k: v for k, v in best_params.items() if v is not None}

# Reinitialize the optimal model 
best_model = SVC(probability=True, random_state=42, **clean_params)

# Train on the training set
best_model.fit(X_train, y_emo_train) 

# Obtain probabilities and select a threshold on the validation set
val_prob = get_positive_prob(best_model, X_val, pos_label=1)


best_threshold = select_threshold(
    labels=y_emo_val, 
    probs=val_prob, 
    min_sensitivity=0.75, 
    min_specificity=0.75,
    pos_label=1
)

print(f"-"*40)
print(f"Selected Threshold on Validation Set: {best_threshold:.4f}")
print(f"-"*40)


# ==========================================
# 4. Final Test Set Evaluation
# ==========================================

# Obtain test set probabilities
test_prob = get_positive_prob(best_model, X_test, pos_label=1)

# Ensure test set labels are in 0/1 format
y_test_bin = (np.asarray(y_emo_test) == 1).astype(int)

# Apply the threshold selected from the validation set
y_test_pred = (test_prob >= best_threshold).astype(int)

# Calculate metrics
test_auc = roc_auc_score(y_test_bin, test_prob)
cm = _safe_confusion_matrix(y_test_bin, y_test_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity_final = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_final = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy_final = accuracy_score(y_test_bin, y_test_pred)

# Output the final result
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {accuracy_final:.4f}")
print(f"Test Sensitivity (Recall): {sensitivity_final:.4f}")
print(f"Test Specificity: {specificity_final:.4f}")
print("Test Confusion Matrix:")
print(cm)
print(f"(Format: [[TN, FP],\n          [FN, TP]])")

In [ ]:
task1_origin = test_prob

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, recall_score

y_true          = np.array(y_emo_test)
y_pred_baseline = np.array(task1_origin)  
y_pred_ours     = np.array(task1_converted) 

threshold_origin = 0.08549339591704199770028083094075554981827735900879
threshold_converted = 0.05998985061791190648294858078770630527287721633911

prob_baseline = y_pred_baseline
prob_ours = y_pred_ours

# 基本检查
assert y_true.ndim == 1
assert prob_baseline.shape == prob_ours.shape == y_true.shape
assert set(np.unique(y_true)).issubset({0, 1}), "y_true must be a binary label 0/1"
assert len(np.unique(y_true)) == 2, "y_true must contain both category 0 and category 1"


# ============================================================
# 2. Inspection direction setting
# ============================================================
# "two-sided" : Only check if they are different, without pre-setting a direction.
# "greater"   : Ours > Baseline
# "less"      : Ours < Baseline


ALT_AUC     = "less"
ALT_REC_C1  = "less"
ALT_REC_C0  = "less"


# ============================================================
# 3. Bootstrap parameters
# ============================================================
rng = np.random.RandomState(42)
n_boot = 5000


# ============================================================
# 4. Stratified bootstrap
# ============================================================
def stratified_bootstrap_index(y, rng):
    """
    Hierarchical Bootstrap:

    Sampling with replacement is performed for each category, ensuring that each resampling includes all categories,

    and maintaining the same sample size for each category as the original test set.
    """
    idx_parts = []

    for c in np.unique(y):
        idx_c = np.flatnonzero(y == c)
        idx_parts.append(
            rng.choice(idx_c, size=len(idx_c), replace=True)
        )

    idx = np.concatenate(idx_parts, axis=0)
    rng.shuffle(idx)

    return idx


# ============================================================
# 5. Observations: Indicators on the full test set
# ============================================================
pred_b_all = (prob_baseline >= threshold_origin).astype(int)
pred_o_all = (prob_ours >= threshold_converted).astype(int)

obs_auc_b = roc_auc_score(y_true, prob_baseline)
obs_auc_o = roc_auc_score(y_true, prob_ours)

obs_rec_b_c1 = recall_score(
    y_true,
    pred_b_all,
    pos_label=1,
    zero_division=0
)

obs_rec_o_c1 = recall_score(
    y_true,
    pred_o_all,
    pos_label=1,
    zero_division=0
)

obs_rec_b_c0 = recall_score(
    y_true,
    pred_b_all,
    pos_label=0,
    zero_division=0
)

obs_rec_o_c0 = recall_score(
    y_true,
    pred_o_all,
    pos_label=0,
    zero_division=0
)


# ============================================================
# 6. Paired Bootstrap
# ============================================================
boot_auc_b = []
boot_auc_o = []

boot_rec_b_c1 = []
boot_rec_o_c1 = []

boot_rec_b_c0 = []
boot_rec_o_c0 = []

print("running Paired Bootstrap ...")

for _ in range(n_boot):
    idx = stratified_bootstrap_index(y_true, rng)

    yb = y_true[idx]
    pb = prob_baseline[idx]
    po = prob_ours[idx]

    # Stratified sampling usually includes two categories, so let's double-check this to be on the safe side.
    if len(np.unique(yb)) < 2:
        continue

    # ---------- AUC ----------
    boot_auc_b.append(roc_auc_score(yb, pb))
    boot_auc_o.append(roc_auc_score(yb, po))

    # ---------- Thresholded predictions ----------
    pred_b = (pb >= threshold_origin).astype(int)
    pred_o = (po >= threshold_converted).astype(int)

    # ---------- Recall class 1 ----------
    boot_rec_b_c1.append(
        recall_score(yb, pred_b, pos_label=1, zero_division=0)
    )
    boot_rec_o_c1.append(
        recall_score(yb, pred_o, pos_label=1, zero_division=0)
    )

    # ---------- Recall class 0 ----------
    boot_rec_b_c0.append(
        recall_score(yb, pred_b, pos_label=0, zero_division=0)
    )
    boot_rec_o_c0.append(
        recall_score(yb, pred_o, pos_label=0, zero_division=0)
    )


boot_auc_b = np.asarray(boot_auc_b, dtype=float)
boot_auc_o = np.asarray(boot_auc_o, dtype=float)

boot_rec_b_c1 = np.asarray(boot_rec_b_c1, dtype=float)
boot_rec_o_c1 = np.asarray(boot_rec_o_c1, dtype=float)

boot_rec_b_c0 = np.asarray(boot_rec_b_c0, dtype=float)
boot_rec_o_c0 = np.asarray(boot_rec_o_c0, dtype=float)

print(f" Bootstrap sample size: {len(boot_auc_b)} / {n_boot}")
print("=" * 75)


# ============================================================
# 7. Unified output function
# ============================================================
def report_metric(
    name,
    obs_b,
    obs_o,
    boot_b,
    boot_o,
    alternative="two-sided"
):

    boot_b = np.asarray(boot_b, dtype=float)
    boot_o = np.asarray(boot_o, dtype=float)

    if len(boot_b) == 0 or len(boot_o) == 0:
        print(f"{name}: No valid bootstrap samples.")
        print("-" * 75)
        return

    if len(boot_b) != len(boot_o):
        raise ValueError("boot_b , boot_o have different length.")

    diff = boot_o - boot_b
    obs_diff = obs_o - obs_b

    ci_b = np.percentile(boot_b, [2.5, 97.5])
    ci_o = np.percentile(boot_o, [2.5, 97.5])

    print(name)
    print(f"  Baseline: {obs_b:.2f}   95% CI [{ci_b[0]:.2f}, {ci_b[1]:.2f}]")
    print(f"  Ours:     {obs_o:.2f}   95% CI [{ci_o[0]:.2f}, {ci_o[1]:.2f}]")
    print(f"  Δ Ours - Baseline = {obs_diff:+.2f}")

    # ------------------------------------------------------------
    # two-sided test
    # ------------------------------------------------------------
    if alternative == "two-sided":
        ci_diff = np.percentile(diff, [2.5, 97.5])

        if np.isclose(obs_diff, 0.0, atol=1e-12):
            p_value = 1.0
        elif obs_diff > 0:
            tail_count = np.sum(diff <= 0)
            p_value = min(1.0, 2.0 * (tail_count + 1) / (len(diff) + 1))
        else:
            tail_count = np.sum(diff >= 0)
            p_value = min(1.0, 2.0 * (tail_count + 1) / (len(diff) + 1))

        significant = (ci_diff[0] > 0) or (ci_diff[1] < 0)

        print(f"  Two-sided 95% CI of Δ: [{ci_diff[0]:+.2f}, {ci_diff[1]:+.2f}]")
        print(f"  two-sided p ≈ {p_value:.2f}")

        if significant:
            print("   Significant on both sides: The 95% CI of Δ does not include 0.")
        else:
            print("   Not significant on both sides: 95% CI for Δ includes 0.")

    # ------------------------------------------------------------
    # single-side：Ours > Baseline
    # ------------------------------------------------------------
    elif alternative == "greater":
        lower_bound = np.percentile(diff, 5)

        p_value = (np.sum(diff <= 0) + 1) / (len(diff) + 1)

        significant = lower_bound > 0

        print("  One-sided hypothesis: Ours > Baseline")
        print(f"  One-sided 95% lower bound of Δ = {lower_bound:+.4f}")
        print(f"  one-sided p ≈ {p_value:.4f}")

        if significant:
            print("   One-sided significance: Ours is significantly greater than Baseline.")
        else:
            print("   One-sided insignificant.")

    # ------------------------------------------------------------
    # One-sided test: Ours < Baseline
    # ------------------------------------------------------------
    elif alternative == "less":
        upper_bound = np.percentile(diff, 95)

        p_value = (np.sum(diff >= 0) + 1) / (len(diff) + 1)

        significant = upper_bound < 0

        print("  One-sided hypothesis: Ours < Baseline")
        print(f"  One-sided 95% upper bound of Δ = {upper_bound:+.4f}")
        print(f"  one-sided p ≈ {p_value:.4f}")

        if significant:
            print("   One-sided significance: Ours is significantly smaller than Baseline.")
        else:
            print("   One-sided insignificant.")

    else:
        raise ValueError("alternative can only be set to 'two-sided', 'greater' 或 'less'。")

    print("-" * 75)


# ============================================================
# 8. Output
# ============================================================
print("\nPaired Bootstrap results")
print("=" * 75)

report_metric(
    name="AUC",
    obs_b=obs_auc_b,
    obs_o=obs_auc_o,
    boot_b=boot_auc_b,
    boot_o=boot_auc_o,
    alternative=ALT_AUC
)

report_metric(
    name="Recall (Class 1)",
    obs_b=obs_rec_b_c1,
    obs_o=obs_rec_o_c1,
    boot_b=boot_rec_b_c1,
    boot_o=boot_rec_o_c1,
    alternative=ALT_REC_C1
)

report_metric(
    name="Recall (Class 0)",
    obs_b=obs_rec_b_c0,
    obs_o=obs_rec_o_c0,
    boot_b=boot_rec_b_c0,
    boot_o=boot_rec_o_c0,
    alternative=ALT_REC_C0
)


### Task 1b_Converted_SVM

In [ ]:
import pandas as pd
df8918_egemaps_all = pd.read_pickle('****.pkl')
df8918_egemaps_all

In [ ]:
# read pickle file
import pandas as pd
df8918_egemaps = pd.read_pickle('****.pkl')
df8918_egemaps

In [ ]:
# only keep the 'filename_sid' in  df8918_egemaps_all that in df8918_egemaps
df8918_egemaps_all_filtered = df8918_egemaps_all[df8918_egemaps_all['filename_sid'].isin(df8918_egemaps['filename_sid'])]
df8918_egemaps_all_filtered

In [ ]:
df8918_egemaps = df8918_egemaps_all_filtered.dropna()
df8918_egemaps

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import roc_curve, accuracy_score, roc_auc_score, confusion_matrix
import numpy as np
import random
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df_copd_asthma_speech = df8918_egemaps.copy()

# features, emotion_labels, speaker_ids Defined
features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values


In [ ]:
le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)


unique_speakers = np.unique(speaker_labels)
speaker_to_idx = {spk: [] for spk in unique_speakers}
for i, spk in enumerate(speaker_labels):
    speaker_to_idx[spk].append(i)

# Step 1: Divide the dataset into test sets (20%)
train_val_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)
# Step 2: Divide the remaining 80% into training and validation sets (e.g., 80%/20% -> 64%/16%).
train_speakers, val_speakers = train_test_split(train_val_speakers, test_size=0.2, random_state=42)

# Get Index
train_indices = [i for spk in train_speakers for i in speaker_to_idx[spk]]
val_indices = [i for spk in val_speakers for i in speaker_to_idx[spk]]
test_indices = [i for spk in test_speakers for i in speaker_to_idx[spk]]

X_train, y_emo_train, y_spk_train = features[train_indices], emotion_labels[train_indices], speaker_labels[train_indices]
X_val, y_emo_val, y_spk_val = features[val_indices], emotion_labels[val_indices], speaker_labels [val_indices]
X_test, y_emo_test, y_spk_test = features[test_indices], emotion_labels[test_indices], speaker_labels[test_indices]


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)



In [ ]:
# SVM Model Definition
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [2, 3, 4],
    'class_weight': [None]
}

svm_model = svm.SVC(probability=True)

# Use the validation set for parameter tuning (no longer use cross-validation).
best_score = -np.inf
best_params = None
best_model = None

print("Starting manual hyperparameter tuning on validation set...")
for C in param_grid['C']:
    for kernel in param_grid['kernel']:
        #for gamma in param_grid['gamma']:
            if kernel == 'poly':
                for degree in param_grid['degree']:
                    #for coef0 in param_grid['coef0']:
                        for class_weight in param_grid['class_weight']:
                            try:
                                model = svm.SVC(
                                    C=C, kernel=kernel, degree=degree, class_weight=class_weight, probability=True, random_state=42
                                )
                                model.fit(X_train, y_emo_train)
                                val_prob = model.predict_proba(X_val)[:, 1]
                                auc = roc_auc_score(y_emo_val, val_prob)
                                if auc > best_score:
                                    best_score = auc
                                    best_params = {
                                        'C': C, 'kernel': kernel, 
                                        'degree': degree,  'class_weight': class_weight
                                    }
                                    best_model = model
                            except:
                                continue
            else:
                # For non-poly cores, degree and coef0 are not required (or are fixed)
                for class_weight in param_grid['class_weight']:
                    try:
                        model = svm.SVC(
                            C=C, kernel=kernel, class_weight=class_weight,
                            probability=True, random_state=42
                        )
                        model.fit(X_train, y_emo_train)
                        val_prob = model.predict_proba(X_val)[:, 1]
                        auc = roc_auc_score(y_emo_val, val_prob)
                        if auc > best_score:
                            best_score = auc
                            best_params = {
                                'C': C, 'kernel': kernel, 
                                'degree': None, 'class_weight': class_weight
                            }
                            best_model = model
                    except:
                        continue

print(f"Best Parameters (selected on validation set): {best_params}")
print(f"Best Validation AUC: {best_score:.4f}")

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score

# ==========================================
# 1. Definition of Functions
# ==========================================

def get_positive_prob(model, X, pos_label=1):
    """Securely obtain the predicted probability of the positive class, avoiding errors caused by the class label order not being [0, 1]."""
    classes = list(model.classes_)
    if pos_label not in classes:
        raise ValueError(f"pos_label={pos_label} not found in model.classes_: {classes}")
    pos_idx = classes.index(pos_label)
    return model.predict_proba(X)[:, pos_idx]

def _safe_confusion_matrix(y_true, y_pred):
    """Force labels=[0, 1] to prevent ravel() from throwing an error when the prediction is all 0 or all 1."""
    return confusion_matrix(y_true, y_pred, labels=[0, 1])

# ==========================================
# 2. The select_threshold function you provided
# ==========================================

def select_threshold(labels, probs, min_sensitivity=0.75, min_specificity=0.75, pos_label=1):
    labels = np.asarray(labels)
    probs = np.asarray(probs, dtype=np.float64)

    # Filter out NaN/INF
    valid_prob = np.isfinite(probs)
    labels = labels[valid_prob]
    probs = probs[valid_prob]

    # Convert labels to 0/1 for binary classification
    y = (labels == pos_label).astype(np.int64)

    if len(y) == 0 or len(np.unique(y)) < 2:
        print("Warning: Validation set invalid or single-class. Using 0.5.")
        return 0.5

    _, _, thresholds = roc_curve(y, probs)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    thresholds = thresholds[np.isfinite(thresholds)]
    
    # Add boundaries and restrict to [0, 1]
    thresholds = np.unique(np.concatenate([thresholds, np.array([0.0, 1.0])]))
    thresholds = thresholds[(thresholds >= 0.0) & (thresholds <= 1.0)]

    f1_scores, sens_scores, spec_scores = [], [], []

    for thresh in thresholds:
        preds = (probs >= thresh).astype(np.int64)
        f1_scores.append(float(f1_score(y, preds, zero_division=0)))
        
        cm = _safe_confusion_matrix(y, preds)
        tn, fp, fn, tp = cm.ravel()
        sens_scores.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        spec_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    # Logic 1: Simultaneously satisfy Sensitivity and Spec constraints
    valid_indices = [i for i in range(len(thresholds)) 
                     if sens_scores[i] >= min_sensitivity and spec_scores[i] >= min_specificity]

    if valid_indices:
        best_idx = valid_indices[int(np.argmax([f1_scores[i] for i in valid_indices]))]
    else:
        # Logic 2: As a fallback, only satisfy Sensitivity
        valid_sens_indices = [i for i, s in enumerate(sens_scores) if s >= min_sensitivity]
        if valid_sens_indices:
            print("Warning: Fallback to Sensitivity-only constraint.")
            best_idx = valid_sens_indices[int(np.argmax([f1_scores[i] for i in valid_sens_indices]))]
        else:
            # Logic 3: Maximize global F1
            print("Warning: Fallback to global best F1 threshold.")
            best_idx = int(np.argmax(f1_scores))

    return float(thresholds[best_idx])


# ==========================================
# 3. Model Preparation and Threshold Selection
# ==========================================

# Your optimal parameters
best_params = {'C': 0.1, 'kernel': 'linear', 'degree': None, 'class_weight': None}

# Clean up parameters with values ​​of None to prevent SVC initialization errors
clean_params = {k: v for k, v in best_params.items() if v is not None}

# Reinitialize the optimal model (if your previous loop's `best_model` is still in memory, you can use `best_model` directly)
best_model = SVC(probability=True, random_state=42, **clean_params)

# Train on the training set (assuming `X_train` and `y_emo_train` have been processed by StandardScaler)
best_model.fit(X_train, y_emo_train) 

# Obtain probabilities and select a threshold on the validation set
val_prob = get_positive_prob(best_model, X_val, pos_label=1)

best_threshold = select_threshold(
    labels=y_emo_val, 
    probs=val_prob, 
    min_sensitivity=0.45, 
    min_specificity=0.3,
    pos_label=1
)

print(f"-"*40)
print(f"Selected Threshold on Validation Set: {best_threshold:.4f}")
print(f"-"*40)


# ==========================================
# 4. Final Test Set Evaluation
# ==========================================

# Obtain test set probabilities
test_prob = get_positive_prob(best_model, X_test, pos_label=1)

# Ensure test set labels are in 0/1 format
y_test_bin = (np.asarray(y_emo_test) == 1).astype(int)

# Apply the threshold selected from the validation set
y_test_pred = (test_prob >= best_threshold).astype(int)

# Calculate metrics
test_auc = roc_auc_score(y_test_bin, test_prob)
cm = _safe_confusion_matrix(y_test_bin, y_test_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity_final = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_final = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy_final = accuracy_score(y_test_bin, y_test_pred)

# Output the final result
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {accuracy_final:.4f}")
print(f"Test Sensitivity (Recall): {sensitivity_final:.4f}")
print(f"Test Specificity: {specificity_final:.4f}")
print("Test Confusion Matrix:")
print(cm)
print(f"(Format: [[TN, FP],\n          [FN, TP]])")

In [ ]:
#print best model parameters
print(f"Best Model Parameters: {clean_params}")

In [ ]:
print(f"\n Optimal threshold({best_threshold:.50f}):")

In [ ]:
task1_converted = test_prob

### Task1b _ None_converted SVM

In [ ]:
# read pickle file
import pandas as pd

df8918_all = pd.read_pickle('******.pkl')
df8918_all

In [ ]:
# read pickle file
import pandas as pd
df8918_egemaps = pd.read_pickle('******.pkl')
df8918_egemaps

In [ ]:
# keep the rows that 'filename_sid' in df8918_egemaps_all that are also in df8918_egemaps
df8918_egemaps_all_filtered = df8918_all[df8918_all['filename_sid'].isin(df8918_egemaps['filename_sid'])]
df8918_egemaps_all_filtered

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import roc_curve, accuracy_score, roc_auc_score, confusion_matrix
import numpy as np
import random
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

df_copd_asthma_speech = df8918_egemaps_all_filtered.copy()

# features, emotion_labels, speaker_ids defined
features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['event_clinical_yesno'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values


In [ ]:
le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)


unique_speakers = np.unique(speaker_labels)
speaker_to_idx = {spk: [] for spk in unique_speakers}
for i, spk in enumerate(speaker_labels):
    speaker_to_idx[spk].append(i)

# Step 1: Divide the dataset into test sets (20%)
train_val_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)
# Step 2: Divide the remaining 80% into training and validation sets (e.g., 80%/20% -> 64%/16%).
train_speakers, val_speakers = train_test_split(train_val_speakers, test_size=0.2, random_state=42)

# Get Index
train_indices = [i for spk in train_speakers for i in speaker_to_idx[spk]]
val_indices = [i for spk in val_speakers for i in speaker_to_idx[spk]]
test_indices = [i for spk in test_speakers for i in speaker_to_idx[spk]]

X_train, y_emo_train, y_spk_train = features[train_indices], emotion_labels[train_indices], speaker_labels[train_indices]
X_val, y_emo_val, y_spk_val = features[val_indices], emotion_labels[val_indices], speaker_labels [val_indices]
X_test, y_emo_test, y_spk_test = features[test_indices], emotion_labels[test_indices], speaker_labels[test_indices]


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)



In [ ]:
# SVM Model Definition
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [2, 3, 4],
    'class_weight': [None]
}

svm_model = svm.SVC(probability=True)

# Use the validation set for parameter tuning (no longer use cross-validation).
best_score = -np.inf
best_params = None
best_model = None

print("Starting manual hyperparameter tuning on validation set...")
for C in param_grid['C']:
    for kernel in param_grid['kernel']:
        #for gamma in param_grid['gamma']:
            if kernel == 'poly':
                for degree in param_grid['degree']:
                    #for coef0 in param_grid['coef0']:
                        for class_weight in param_grid['class_weight']:
                            try:
                                model = svm.SVC(
                                    C=C, kernel=kernel, degree=degree, class_weight=class_weight, probability=True, random_state=42
                                )
                                model.fit(X_train, y_emo_train)
                                val_prob = model.predict_proba(X_val)[:, 1]
                                auc = roc_auc_score(y_emo_val, val_prob)
                                if auc > best_score:
                                    best_score = auc
                                    best_params = {
                                        'C': C, 'kernel': kernel, 
                                        'degree': degree,  'class_weight': class_weight
                                    }
                                    best_model = model
                            except:
                                continue
            else:
                # For non-poly cores, degree and coef0 are not required (or are fixed).
                for class_weight in param_grid['class_weight']:
                    try:
                        model = svm.SVC(
                            C=C, kernel=kernel, class_weight=class_weight,
                            probability=True, random_state=42
                        )
                        model.fit(X_train, y_emo_train)
                        val_prob = model.predict_proba(X_val)[:, 1]
                        auc = roc_auc_score(y_emo_val, val_prob)
                        if auc > best_score:
                            best_score = auc
                            best_params = {
                                'C': C, 'kernel': kernel, 
                                'degree': None, 'class_weight': class_weight
                            }
                            best_model = model
                    except:
                        continue

print(f"Best Parameters (selected on validation set): {best_params}")
print(f"Best Validation AUC: {best_score:.4f}")

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score

# ==========================================
# 1. Definition of Functions
# ==========================================

def get_positive_prob(model, X, pos_label=1):
    """Securely obtain the predicted probability of the positive class, avoiding errors caused by the class label order not being [0, 1]."""
    classes = list(model.classes_)
    if pos_label not in classes:
        raise ValueError(f"pos_label={pos_label} not found in model.classes_: {classes}")
    pos_idx = classes.index(pos_label)
    return model.predict_proba(X)[:, pos_idx]

def _safe_confusion_matrix(y_true, y_pred):
    """Force labels=[0, 1] to prevent ravel() from throwing an error when the prediction is all 0 or all 1."""
    return confusion_matrix(y_true, y_pred, labels=[0, 1])

# ==========================================
# 2. The select_threshold function you provided
# ==========================================

def select_threshold(labels, probs, min_sensitivity=0.75, min_specificity=0.75, pos_label=1):
    labels = np.asarray(labels)
    probs = np.asarray(probs, dtype=np.float64)

    # Filter out NaN/INF
    valid_prob = np.isfinite(probs)
    labels = labels[valid_prob]
    probs = probs[valid_prob]

    # Convert labels to 0/1 for binary classification
    y = (labels == pos_label).astype(np.int64)

    if len(y) == 0 or len(np.unique(y)) < 2:
        print("Warning: Validation set invalid or single-class. Using 0.5.")
        return 0.5

    _, _, thresholds = roc_curve(y, probs)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    thresholds = thresholds[np.isfinite(thresholds)]
    
    # Add boundaries and restrict to [0, 1]
    thresholds = np.unique(np.concatenate([thresholds, np.array([0.0, 1.0])]))
    thresholds = thresholds[(thresholds >= 0.0) & (thresholds <= 1.0)]

    f1_scores, sens_scores, spec_scores = [], [], []

    for thresh in thresholds:
        preds = (probs >= thresh).astype(np.int64)
        f1_scores.append(float(f1_score(y, preds, zero_division=0)))
        
        cm = _safe_confusion_matrix(y, preds)
        tn, fp, fn, tp = cm.ravel()
        sens_scores.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        spec_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    # Logic 1: Simultaneously satisfy Sensitivity and Spec constraints
    valid_indices = [i for i in range(len(thresholds)) 
                     if sens_scores[i] >= min_sensitivity and spec_scores[i] >= min_specificity]

    if valid_indices:
        best_idx = valid_indices[int(np.argmax([f1_scores[i] for i in valid_indices]))]
    else:
        # Logic 2: As a fallback, only satisfy Sensitivity
        valid_sens_indices = [i for i, s in enumerate(sens_scores) if s >= min_sensitivity]
        if valid_sens_indices:
            print("Warning: Fallback to Sensitivity-only constraint.")
            best_idx = valid_sens_indices[int(np.argmax([f1_scores[i] for i in valid_sens_indices]))]
        else:
            # Logic 3: Maximize global F1
            print("Warning: Fallback to global best F1 threshold.")
            best_idx = int(np.argmax(f1_scores))

    return float(thresholds[best_idx])


# ==========================================
# 3. Model Preparation and Threshold Selection
# ==========================================

# Your optimal parameters
best_params = {'C': 1, 'kernel': 'rbf', 'degree': None, 'class_weight': None}

#  Clean up parameters with values ​​of None to prevent SVC initialization errors
clean_params = {k: v for k, v in best_params.items() if v is not None}

# Reinitialize the optimal model (if your previous loop's `best_model` is still in memory, you can use `best_model` directly)
best_model = SVC(probability=True, random_state=42, **clean_params)

# Train on the training set (assuming `X_train` and `y_emo_train` have been processed by StandardScaler)
best_model.fit(X_train, y_emo_train) 

# Obtain probabilities and select a threshold on the validation set
val_prob = get_positive_prob(best_model, X_val, pos_label=1)

best_threshold = select_threshold(
    labels=y_emo_val, 
    probs=val_prob, 
    min_sensitivity=0.45, 
    min_specificity=0.3,
    pos_label=1
)

print(f"-"*40)
print(f"Selected Threshold on Validation Set: {best_threshold:.4f}")
print(f"-"*40)


# ==========================================
# 4. Final Test Set Evaluation
# ==========================================

# Obtain test set probabilities
test_prob = get_positive_prob(best_model, X_test, pos_label=1)

# Ensure test set labels are in 0/1 format
y_test_bin = (np.asarray(y_emo_test) == 1).astype(int)

# Apply the threshold selected from the validation set
y_test_pred = (test_prob >= best_threshold).astype(int)

# Calculate metrics
test_auc = roc_auc_score(y_test_bin, test_prob)
cm = _safe_confusion_matrix(y_test_bin, y_test_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity_final = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_final = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy_final = accuracy_score(y_test_bin, y_test_pred)

# Output the final result
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {accuracy_final:.4f}")
print(f"Test Sensitivity (Recall): {sensitivity_final:.4f}")
print(f"Test Specificity: {specificity_final:.4f}")
print("Test Confusion Matrix:")
print(cm)
print(f"(Format: [[TN, FP],\n          [FN, TP]])")

In [ ]:
#print best model parameters
print(f"Best Model Parameters: {clean_params}")

In [ ]:
print(f"\n Optimal threshold({best_threshold:.50f}):")

In [ ]:
task1_origin = test_prob

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, recall_score


y_true          = np.array(y_emo_test)
y_pred_baseline = np.array(task1_origin) # 0.11273495721030012128860420261844410561025142669678
y_pred_ours     = np.array(task1_converted)# 0.06935841393061017989829508678667480126023292541504

threshold_origin = 0.36839217088143166867197919600585009902715682983398
threshold_converted = 0.49414072231035627291717560183315072208642959594727


prob_baseline = y_pred_baseline
prob_ours = y_pred_ours

# Basic checks
assert y_true.ndim == 1
assert prob_baseline.shape == prob_ours.shape == y_true.shape
assert set(np.unique(y_true)).issubset({0, 1}), "y_true must be a binary label 0/1"
assert len(np.unique(y_true)) == 2, "y_true must contain both category 0 and category 1"


# ============================================================
# 2. Inspection direction setting
# ============================================================
# 可选值：
# "two-sided" : Only check if they are different, without pre-setting a direction.
# "greater"   :  Ours > Baseline
# "less"      :  Ours < Baseline


ALT_AUC     = "less"
ALT_REC_C1  = "less"
ALT_REC_C0  = "less"


# ============================================================
# 3. Bootstrap parameters
# ============================================================
rng = np.random.RandomState(42)
n_boot = 5000


# ============================================================
# 4. Stratified bootstrap
# ============================================================
def stratified_bootstrap_index(y, rng):
    """
    Hierarchical Bootstrap:

    Sampling with replacement is performed for each category, ensuring that each resampling includes all categories,

    and maintaining the same sample size for each category as the original test set.
    """
    idx_parts = []

    for c in np.unique(y):
        idx_c = np.flatnonzero(y == c)
        idx_parts.append(
            rng.choice(idx_c, size=len(idx_c), replace=True)
        )

    idx = np.concatenate(idx_parts, axis=0)
    rng.shuffle(idx)

    return idx


# ============================================================
# 5. Observations: Indicators on the full test set
# ============================================================
pred_b_all = (prob_baseline >= threshold_origin).astype(int)
pred_o_all = (prob_ours >= threshold_converted).astype(int)

obs_auc_b = roc_auc_score(y_true, prob_baseline)
obs_auc_o = roc_auc_score(y_true, prob_ours)

obs_rec_b_c1 = recall_score(
    y_true,
    pred_b_all,
    pos_label=1,
    zero_division=0
)

obs_rec_o_c1 = recall_score(
    y_true,
    pred_o_all,
    pos_label=1,
    zero_division=0
)

obs_rec_b_c0 = recall_score(
    y_true,
    pred_b_all,
    pos_label=0,
    zero_division=0
)

obs_rec_o_c0 = recall_score(
    y_true,
    pred_o_all,
    pos_label=0,
    zero_division=0
)


# ============================================================
# 6. Paired Bootstrap
# ============================================================
boot_auc_b = []
boot_auc_o = []

boot_rec_b_c1 = []
boot_rec_o_c1 = []

boot_rec_b_c0 = []
boot_rec_o_c0 = []

print("running Paired Bootstrap ...")

for _ in range(n_boot):
    idx = stratified_bootstrap_index(y_true, rng)

    yb = y_true[idx]
    pb = prob_baseline[idx]
    po = prob_ours[idx]

    # Stratified sampling usually includes two categories, so let's double-check this to be on the safe side.
    if len(np.unique(yb)) < 2:
        continue

    # ---------- AUC ----------
    boot_auc_b.append(roc_auc_score(yb, pb))
    boot_auc_o.append(roc_auc_score(yb, po))

    # ---------- Thresholded predictions ----------
    pred_b = (pb >= threshold_origin).astype(int)
    pred_o = (po >= threshold_converted).astype(int)

    # ---------- Recall class 1 ----------
    boot_rec_b_c1.append(
        recall_score(yb, pred_b, pos_label=1, zero_division=0)
    )
    boot_rec_o_c1.append(
        recall_score(yb, pred_o, pos_label=1, zero_division=0)
    )

    # ---------- Recall class 0 ----------
    boot_rec_b_c0.append(
        recall_score(yb, pred_b, pos_label=0, zero_division=0)
    )
    boot_rec_o_c0.append(
        recall_score(yb, pred_o, pos_label=0, zero_division=0)
    )


boot_auc_b = np.asarray(boot_auc_b, dtype=float)
boot_auc_o = np.asarray(boot_auc_o, dtype=float)

boot_rec_b_c1 = np.asarray(boot_rec_b_c1, dtype=float)
boot_rec_o_c1 = np.asarray(boot_rec_o_c1, dtype=float)

boot_rec_b_c0 = np.asarray(boot_rec_b_c0, dtype=float)
boot_rec_o_c0 = np.asarray(boot_rec_o_c0, dtype=float)

print(f" Bootstrap sample size: {len(boot_auc_b)} / {n_boot}")
print("=" * 75)


# ============================================================
# 7. Unified output function
# ============================================================
def report_metric(
    name,
    obs_b,
    obs_o,
    boot_b,
    boot_o,
    alternative="two-sided"
):
    """
    Outputting a single indicator:

    1. Point estimate of Baseline/Ours and 95% CI;

    2. Δ = Ours - Baseline;

    3. Two-sided or one-sided significance test.
    """
    boot_b = np.asarray(boot_b, dtype=float)
    boot_o = np.asarray(boot_o, dtype=float)

    if len(boot_b) == 0 or len(boot_o) == 0:
        print(f"{name}: No valid bootstrap samples.")
        print("-" * 75)
        return

    if len(boot_b) != len(boot_o):
        raise ValueError("boot_b , boot_o, The lengths are inconsistent")

    diff = boot_o - boot_b
    obs_diff = obs_o - obs_b

    ci_b = np.percentile(boot_b, [2.5, 97.5])
    ci_o = np.percentile(boot_o, [2.5, 97.5])

    print(name)
    print(f"  Baseline: {obs_b:.2f}   95% CI [{ci_b[0]:.2f}, {ci_b[1]:.2f}]")
    print(f"  Ours:     {obs_o:.2f}   95% CI [{ci_o[0]:.2f}, {ci_o[1]:.2f}]")
    print(f"  Δ Ours - Baseline = {obs_diff:+.2f}")

    # ------------------------------------------------------------
    # two-sided test
    # ------------------------------------------------------------
    if alternative == "two-sided":
        ci_diff = np.percentile(diff, [2.5, 97.5])

        if np.isclose(obs_diff, 0.0, atol=1e-12):
            p_value = 1.0
        elif obs_diff > 0:
            tail_count = np.sum(diff <= 0)
            p_value = min(1.0, 2.0 * (tail_count + 1) / (len(diff) + 1))
        else:
            tail_count = np.sum(diff >= 0)
            p_value = min(1.0, 2.0 * (tail_count + 1) / (len(diff) + 1))

        significant = (ci_diff[0] > 0) or (ci_diff[1] < 0)

        print(f"  Two-sided 95% CI of Δ: [{ci_diff[0]:+.2f}, {ci_diff[1]:+.2f}]")
        print(f"  two-sided p ≈ {p_value:.2f}")

        if significant:
            print("   Significant on both sides: The 95% CI of Δ does not include 0")
        else:
            print("   Not significant on both sides: 95% CI for Δ includes 0.")

    # ------------------------------------------------------------
    # single-side：Ours > Baseline
    # ------------------------------------------------------------
    elif alternative == "greater":
        lower_bound = np.percentile(diff, 5)

        p_value = (np.sum(diff <= 0) + 1) / (len(diff) + 1)

        significant = lower_bound > 0

        print("  One-sided hypothesis: Ours > Baseline")
        print(f"  One-sided 95% lower bound of Δ = {lower_bound:+.4f}")
        print(f"  one-sided p ≈ {p_value:.4f}")

        if significant:
            print("   One-sided significance: Ours is significantly greater than Baseline.")
        else:
            print("   One-sided insignificant.")

    # ------------------------------------------------------------
    # single-side test：Ours < Baseline
    # ------------------------------------------------------------
    elif alternative == "less":
        upper_bound = np.percentile(diff, 95)

        p_value = (np.sum(diff >= 0) + 1) / (len(diff) + 1)

        significant = upper_bound < 0

        print("  One-sided hypothesis: Ours < Baseline")
        print(f"  One-sided 95% upper bound of Δ = {upper_bound:+.4f}")
        print(f"  one-sided p ≈ {p_value:.4f}")

        if significant:
            print("   One-sided significance: Ours is significantly smaller than Baseline.")
        else:
            print("   One-sided insignificant.")

    else:
        raise ValueError("alternative can be only 'two-sided', 'greater' or 'less'。")

    print("-" * 75)


# ============================================================
# 8. Output
# ============================================================
print("\nPaired Bootstrap results")
print("=" * 75)

report_metric(
    name="AUC",
    obs_b=obs_auc_b,
    obs_o=obs_auc_o,
    boot_b=boot_auc_b,
    boot_o=boot_auc_o,
    alternative=ALT_AUC
)

report_metric(
    name="Recall (Class 1)",
    obs_b=obs_rec_b_c1,
    obs_o=obs_rec_o_c1,
    boot_b=boot_rec_b_c1,
    boot_o=boot_rec_o_c1,
    alternative=ALT_REC_C1
)

report_metric(
    name="Recall (Class 0)",
    obs_b=obs_rec_b_c0,
    obs_o=obs_rec_o_c0,
    boot_b=boot_rec_b_c0,
    boot_o=boot_rec_o_c0,
    alternative=ALT_REC_C0
)



### Task 3_Converted_SVM

In [ ]:
import pandas as pd

# 加载数据
df_copd_asthma_speech = pd.read_pickle('******.pkl')
df_copd_asthma_speech

In [ ]:
df_copd_asthma_speech['filename_sid'] = df_copd_asthma_speech['sid'].astype(str) + '_' + df_copd_asthma_speech['filename']
df_copd_asthma_speech

In [ ]:
# read pickle file
import pandas as pd
df8918_egemaps = pd.read_pickle('******.pkl')
df8918_egemaps

In [ ]:
df8918_egemaps = df8918_egemaps.dropna()
df8918_egemaps

In [ ]:
# only keep the rows in df8918_egemaps that are also in df_copd_asthma_speech based on 'filename_sid'
df8918_egemaps_keep = df8918_egemaps[df8918_egemaps['filename_sid'].isin(df_copd_asthma_speech['filename_sid'])]
df8918_egemaps_keep


In [ ]:
import numpy as np
# features, emotion_labels, speaker_ids defined
df_copd_asthma_speech = df8918_egemaps_keep
features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['label'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)


unique_speakers = np.unique(speaker_labels)
speaker_to_idx = {spk: [] for spk in unique_speakers}
for i, spk in enumerate(speaker_labels):
    speaker_to_idx[spk].append(i)

# Step 1: Divide the dataset into test sets (20%)
train_val_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=0)
# Step 2: Divide the remaining 80% into training and validation sets (e.g., 80%/20% -> 64%/16%).
train_speakers, val_speakers = train_test_split(train_val_speakers, test_size=0.2, random_state=0)

# Get Index
train_indices = [i for spk in train_speakers for i in speaker_to_idx[spk]]
val_indices = [i for spk in val_speakers for i in speaker_to_idx[spk]]
test_indices = [i for spk in test_speakers for i in speaker_to_idx[spk]]

X_train, y_emo_train, y_spk_train = features[train_indices], emotion_labels[train_indices], speaker_labels[train_indices]
X_val, y_emo_val, y_spk_val = features[val_indices], emotion_labels[val_indices], speaker_labels[val_indices]
X_test, y_emo_test, y_spk_test = features[test_indices], emotion_labels[test_indices], speaker_labels[test_indices]


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score

# ==========================================
# 1. Definition of Functions
# ==========================================

def get_positive_prob(model, X, pos_label=1):
    """Securely obtain the predicted probability of the positive class, avoiding errors caused by the class label order not being [0, 1]."""
    classes = list(model.classes_)
    if pos_label not in classes:
        raise ValueError(f"pos_label={pos_label} not found in model.classes_: {classes}")
    pos_idx = classes.index(pos_label)
    return model.predict_proba(X)[:, pos_idx]

def _safe_confusion_matrix(y_true, y_pred):
    """Force labels=[0, 1] to prevent ravel() from throwing an error when the prediction is all 0 or all 1."""
    return confusion_matrix(y_true, y_pred, labels=[0, 1])

# ==========================================
# 2. The select_threshold function you provided
# ==========================================

def select_threshold(labels, probs, min_sensitivity=0.75, min_specificity=0.75, pos_label=1):
    labels = np.asarray(labels)
    probs = np.asarray(probs, dtype=np.float64)

    # Filter out NaN/INF
    valid_prob = np.isfinite(probs)
    labels = labels[valid_prob]
    probs = probs[valid_prob]

    # Convert labels to 0/1 for binary classification
    y = (labels == pos_label).astype(np.int64)

    if len(y) == 0 or len(np.unique(y)) < 2:
        print("Warning: Validation set invalid or single-class. Using 0.5.")
        return 0.5

    _, _, thresholds = roc_curve(y, probs)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    thresholds = thresholds[np.isfinite(thresholds)]
    
    # Add boundaries and restrict to [0, 1]
    thresholds = np.unique(np.concatenate([thresholds, np.array([0.0, 1.0])]))
    thresholds = thresholds[(thresholds >= 0.0) & (thresholds <= 1.0)]

    f1_scores, sens_scores, spec_scores = [], [], []

    for thresh in thresholds:
        preds = (probs >= thresh).astype(np.int64)
        f1_scores.append(float(f1_score(y, preds, zero_division=0)))
        
        cm = _safe_confusion_matrix(y, preds)
        tn, fp, fn, tp = cm.ravel()
        sens_scores.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        spec_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    # Logic 1: Simultaneously satisfy Sensitivity and Spec constraints
    valid_indices = [i for i in range(len(thresholds)) 
                     if sens_scores[i] >= min_sensitivity and spec_scores[i] >= min_specificity]

    if valid_indices:
        best_idx = valid_indices[int(np.argmax([f1_scores[i] for i in valid_indices]))]
    else:
        # Logic 2: As a fallback, only satisfy Sensitivity
        valid_sens_indices = [i for i, s in enumerate(sens_scores) if s >= min_sensitivity]
        if valid_sens_indices:
            print("Warning: Fallback to Sensitivity-only constraint.")
            best_idx = valid_sens_indices[int(np.argmax([f1_scores[i] for i in valid_sens_indices]))]
        else:
            # Logic 3: Maximize global F1
            print("Warning: Fallback to global best F1 threshold.")
            best_idx = int(np.argmax(f1_scores))

    return float(thresholds[best_idx])


# ==========================================
# 3. Model Preparation and Threshold Selection
# ==========================================

# Your optimal parameters
best_params = {'C': 0.1, 'kernel': 'poly', 'degree': 4, 'class_weight': None}

# Clean up parameters with values ​​of None to prevent SVC initialization errors
clean_params = {k: v for k, v in best_params.items() if v is not None}

# Reinitialize the optimal model (if your previous loop's `best_model` is still in memory, you can use `best_model` directly)
best_model = SVC(probability=True, random_state=42, **clean_params)

# Train on the training set (assuming `X_train` and `y_emo_train` have been processed by StandardScaler)
best_model.fit(X_train, y_emo_train) 

# Obtain probabilities and select a threshold on the validation set
val_prob = get_positive_prob(best_model, X_val, pos_label=1)

best_threshold = select_threshold(
    labels=y_emo_val, 
    probs=val_prob, 
    min_sensitivity=0.7, 
    min_specificity=0.7,
    pos_label=1
)

print(f"-"*40)
print(f"Selected Threshold on Validation Set: {best_threshold:.4f}")
print(f"-"*40)


# ==========================================
# 4. Final Test Set Evaluation
# ==========================================

# Obtain test set probabilities
test_prob = get_positive_prob(best_model, X_test, pos_label=1)

# Ensure test set labels are in 0/1 format
y_test_bin = (np.asarray(y_emo_test) == 1).astype(int)

# Apply the threshold selected from the validation set
y_test_pred = (test_prob >= best_threshold).astype(int)

# Calculate metrics
test_auc = roc_auc_score(y_test_bin, test_prob)
cm = _safe_confusion_matrix(y_test_bin, y_test_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity_final = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_final = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy_final = accuracy_score(y_test_bin, y_test_pred)

# Output the final result
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {accuracy_final:.4f}")
print(f"Test Sensitivity (Recall): {sensitivity_final:.4f}")
print(f"Test Specificity: {specificity_final:.4f}")
print("Test Confusion Matrix:")
print(cm)
print(f"(Format: [[TN, FP],\n          [FN, TP]])")

In [ ]:
#print best model parameters
print(f"Best Model Parameters: {clean_params}")

In [ ]:
print(f"\n best_threshold  ({best_threshold:.50f}):")

In [ ]:
task1_converted = test_prob

### Task 3_None_Converted_SVM

In [ ]:
import pandas as pd

# load data
df_copd_asthma_speech = pd.read_pickle('******.pkl')
df_copd_asthma_speech

In [ ]:
import numpy as np
# features, emotion_labels, speaker_ids defined
df_copd_asthma_speech = df_copd_asthma_speech
features = df_copd_asthma_speech.iloc[:, :54].values.astype(np.float32)
emotion_labels = df_copd_asthma_speech['disease'].values.astype(np.int64)
speaker_ids = df_copd_asthma_speech['sid'].values

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

le_speaker = LabelEncoder()
speaker_labels = le_speaker.fit_transform(speaker_ids).astype(np.int64)
num_speakers = len(le_speaker.classes_)


unique_speakers = np.unique(speaker_labels)
speaker_to_idx = {spk: [] for spk in unique_speakers}
for i, spk in enumerate(speaker_labels):
    speaker_to_idx[spk].append(i)

# Step 1: Divide the dataset into test sets (20%)
train_val_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=0)
# Step 2: Divide the remaining 80% into training and validation sets (e.g., 80%/20% -> 64%/16%).
train_speakers, val_speakers = train_test_split(train_val_speakers, test_size=0.2, random_state=0)

# Get Index
train_indices = [i for spk in train_speakers for i in speaker_to_idx[spk]]
val_indices = [i for spk in val_speakers for i in speaker_to_idx[spk]]
test_indices = [i for spk in test_speakers for i in speaker_to_idx[spk]]

X_train, y_emo_train, y_spk_train = features[train_indices], emotion_labels[train_indices], speaker_labels[train_indices]
X_val, y_emo_val, y_spk_val = features[val_indices], emotion_labels[val_indices], speaker_labels[val_indices]
X_test, y_emo_test, y_spk_test = features[test_indices], emotion_labels[test_indices], speaker_labels[test_indices]


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score

# ==========================================
# 1. Definition of Functions
# ==========================================

def get_positive_prob(model, X, pos_label=1):
    """Securely obtain the predicted probability of the positive class, avoiding errors caused by the class label order not being [0, 1]."""
    classes = list(model.classes_)
    if pos_label not in classes:
        raise ValueError(f"pos_label={pos_label} not found in model.classes_: {classes}")
    pos_idx = classes.index(pos_label)
    return model.predict_proba(X)[:, pos_idx]

def _safe_confusion_matrix(y_true, y_pred):
    """Force labels=[0, 1] to prevent ravel() from throwing an error when the prediction is all 0 or all 1."""
    return confusion_matrix(y_true, y_pred, labels=[0, 1])

# ==========================================
# 2. The select_threshold function you provided
# ==========================================

def select_threshold(labels, probs, min_sensitivity=0.75, min_specificity=0.75, pos_label=1):
    labels = np.asarray(labels)
    probs = np.asarray(probs, dtype=np.float64)

    #   Filter out NaN/INF
    valid_prob = np.isfinite(probs)
    labels = labels[valid_prob]
    probs = probs[valid_prob]

    #  Convert labels to 0/1 for binary classification
    y = (labels == pos_label).astype(np.int64)

    if len(y) == 0 or len(np.unique(y)) < 2:
        print("Warning: Validation set invalid or single-class. Using 0.5.")
        return 0.5

    _, _, thresholds = roc_curve(y, probs)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    thresholds = thresholds[np.isfinite(thresholds)]
    
    #  Add boundaries and restrict to [0, 1]
    thresholds = np.unique(np.concatenate([thresholds, np.array([0.0, 1.0])]))
    thresholds = thresholds[(thresholds >= 0.0) & (thresholds <= 1.0)]

    f1_scores, sens_scores, spec_scores = [], [], []

    for thresh in thresholds:
        preds = (probs >= thresh).astype(np.int64)
        f1_scores.append(float(f1_score(y, preds, zero_division=0)))
        
        cm = _safe_confusion_matrix(y, preds)
        tn, fp, fn, tp = cm.ravel()
        sens_scores.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        spec_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    # Logic 1: Simultaneously satisfy Sensitivity and Spec constraints
    valid_indices = [i for i in range(len(thresholds)) 
                     if sens_scores[i] >= min_sensitivity and spec_scores[i] >= min_specificity]

    if valid_indices:
        best_idx = valid_indices[int(np.argmax([f1_scores[i] for i in valid_indices]))]
    else:
        # Logic 2: As a fallback, only satisfy Sensitivity
        valid_sens_indices = [i for i, s in enumerate(sens_scores) if s >= min_sensitivity]
        if valid_sens_indices:
            print("Warning: Fallback to Sensitivity-only constraint.")
            best_idx = valid_sens_indices[int(np.argmax([f1_scores[i] for i in valid_sens_indices]))]
        else:
            # Logic 3: Maximize global F1
            print("Warning: Fallback to global best F1 threshold.")
            best_idx = int(np.argmax(f1_scores))

    return float(thresholds[best_idx])


# ==========================================
# 3. Model Preparation and Threshold Selection
# ==========================================

# Your optimal parameters
best_params = {'C': 10, 'kernel': 'poly', 'degree': 4, 'class_weight': None}

# Clean up parameters with values ​​of None to prevent SVC initialization errors
clean_params = {k: v for k, v in best_params.items() if v is not None}

# Reinitialize the optimal model (if your previous loop's `best_model` is still in memory, you can use `best_model` directly)
best_model = SVC(probability=True, random_state=42, **clean_params)

# Train on the training set (assuming `X_train` and `y_emo_train` have been processed by StandardScaler)
best_model.fit(X_train, y_emo_train) 

# Obtain probabilities and select a threshold on the validation set
val_prob = get_positive_prob(best_model, X_val, pos_label=1)

best_threshold = select_threshold(
    labels=y_emo_val, 
    probs=val_prob, 
    min_sensitivity=0.7, 
    min_specificity=0.7,
    pos_label=1
)

print(f"-"*40)
print(f"Selected Threshold on Validation Set: {best_threshold:.4f}")
print(f"-"*40)


# ==========================================
# 4. Final Test Set Evaluation
# ==========================================

# Obtain test set probabilities
test_prob = get_positive_prob(best_model, X_test, pos_label=1)

# Ensure test set labels are in 0/1 format
y_test_bin = (np.asarray(y_emo_test) == 1).astype(int)

# Apply the threshold selected from the validation set
y_test_pred = (test_prob >= best_threshold).astype(int)

# Calculate metrics
test_auc = roc_auc_score(y_test_bin, test_prob)
cm = _safe_confusion_matrix(y_test_bin, y_test_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity_final = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_final = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy_final = accuracy_score(y_test_bin, y_test_pred)

# Output the final result
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Accuracy: {accuracy_final:.4f}")
print(f"Test Sensitivity (Recall): {sensitivity_final:.4f}")
print(f"Test Specificity: {specificity_final:.4f}")
print("Test Confusion Matrix:")
print(cm)
print(f"(Format: [[TN, FP],\n          [FN, TP]])")

In [ ]:
print(f"\n best_threshold ({best_threshold:.50f}):")

In [ ]:
task1_origin = test_prob

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, recall_score

y_true          = np.array(y_emo_test)
y_pred_baseline = np.array(task1_origin) # 0.11273495721030012128860420261844410561025142669678
y_pred_ours     = np.array(task1_converted)# 0.06935841393061017989829508678667480126023292541504

threshold_origin = 0.65465893774771655166944128723116591572761535644531
threshold_converted = 0.55170579529860297451904216359253041446208953857422


prob_baseline = y_pred_baseline
prob_ours = y_pred_ours

# check the shape of the input arrays
assert y_true.ndim == 1
assert prob_baseline.shape == prob_ours.shape == y_true.shape
assert set(np.unique(y_true)).issubset({0, 1}),  #The category in "y_true" must be either 0 or 1
assert len(np.unique(y_true)) == 2, #y_true must contain both category 0 and category 1


# ============================================================
# 2. Paired Bootstrap Alternative Hypothesis
# ============================================================

# "two-sided" : Only check if they are different, without pre-setting a direction.
# "greater"   :  Ours > Baseline
# "less"      :  Ours < Baseline


ALT_AUC     = "less"
ALT_REC_C1  = "less"
ALT_REC_C0  = "less"


# ============================================================
# 3. Bootstrap Parameters
# ============================================================
rng = np.random.RandomState(42)
n_boot = 5000


# ============================================================
# 4. Stratified bootstrap
# ============================================================
def stratified_bootstrap_index(y, rng):
    """
    Stratified bootstrap sampling: For each category, sample with replacement, and then concatenate the indices of all categories to form a new bootstrap sample.

    """
    idx_parts = []

    for c in np.unique(y):
        idx_c = np.flatnonzero(y == c)
        idx_parts.append(
            rng.choice(idx_c, size=len(idx_c), replace=True)
        )

    idx = np.concatenate(idx_parts, axis=0)
    rng.shuffle(idx)

    return idx


# ============================================================
# 5. Observed Metrics
# ============================================================
pred_b_all = (prob_baseline >= threshold_origin).astype(int)
pred_o_all = (prob_ours >= threshold_converted).astype(int)

obs_auc_b = roc_auc_score(y_true, prob_baseline)
obs_auc_o = roc_auc_score(y_true, prob_ours)

obs_rec_b_c1 = recall_score(
    y_true,
    pred_b_all,
    pos_label=1,
    zero_division=0
)

obs_rec_o_c1 = recall_score(
    y_true,
    pred_o_all,
    pos_label=1,
    zero_division=0
)

obs_rec_b_c0 = recall_score(
    y_true,
    pred_b_all,
    pos_label=0,
    zero_division=0
)

obs_rec_o_c0 = recall_score(
    y_true,
    pred_o_all,
    pos_label=0,
    zero_division=0
)


# ============================================================
# 6. Paired Bootstrap
# ============================================================
boot_auc_b = []
boot_auc_o = []

boot_rec_b_c1 = []
boot_rec_o_c1 = []

boot_rec_b_c0 = []
boot_rec_o_c0 = []

print("running Paired Bootstrap ...")

for _ in range(n_boot):
    idx = stratified_bootstrap_index(y_true, rng)

    yb = y_true[idx]
    pb = prob_baseline[idx]
    po = prob_ours[idx]

    # Stratified sampling usually includes two categories, so check this 
    if len(np.unique(yb)) < 2:
        continue

    # ---------- AUC ----------
    boot_auc_b.append(roc_auc_score(yb, pb))
    boot_auc_o.append(roc_auc_score(yb, po))

    # ---------- Thresholded predictions ----------
    pred_b = (pb >= threshold_origin).astype(int)
    pred_o = (po >= threshold_converted).astype(int)

    # ---------- Recall class 1 ----------
    boot_rec_b_c1.append(
        recall_score(yb, pred_b, pos_label=1, zero_division=0)
    )
    boot_rec_o_c1.append(
        recall_score(yb, pred_o, pos_label=1, zero_division=0)
    )

    # ---------- Recall class 0 ----------
    boot_rec_b_c0.append(
        recall_score(yb, pred_b, pos_label=0, zero_division=0)
    )
    boot_rec_o_c0.append(
        recall_score(yb, pred_o, pos_label=0, zero_division=0)
    )


boot_auc_b = np.asarray(boot_auc_b, dtype=float)
boot_auc_o = np.asarray(boot_auc_o, dtype=float)

boot_rec_b_c1 = np.asarray(boot_rec_b_c1, dtype=float)
boot_rec_o_c1 = np.asarray(boot_rec_o_c1, dtype=float)

boot_rec_b_c0 = np.asarray(boot_rec_b_c0, dtype=float)
boot_rec_o_c0 = np.asarray(boot_rec_o_c0, dtype=float)

print(f"有效 Bootstrap 样本数: {len(boot_auc_b)} / {n_boot}")
print("=" * 75)


# ============================================================
# 7.Unified output function
# ============================================================
def report_metric(
    name,
    obs_b,
    obs_o,
    boot_b,
    boot_o,
    alternative="two-sided"
):
    """
    Outputting a single indicator:

    1. Point estimate of Baseline/Ours and 95% CI;

    2. Δ = Ours - Baseline;

    3. Two-sided or one-sided significance test.
    """
    boot_b = np.asarray(boot_b, dtype=float)
    boot_o = np.asarray(boot_o, dtype=float)

    if len(boot_b) == 0 or len(boot_o) == 0:
        print(f"{name}: No valid bootstrap samples.")
        print("-" * 75)
        return

    if len(boot_b) != len(boot_o):
        raise ValueError("boot_b 和 boot_o 长度不一致。")

    diff = boot_o - boot_b
    obs_diff = obs_o - obs_b

    ci_b = np.percentile(boot_b, [2.5, 97.5])
    ci_o = np.percentile(boot_o, [2.5, 97.5])

    print(name)
    print(f"  Baseline: {obs_b:.2f}   95% CI [{ci_b[0]:.2f}, {ci_b[1]:.2f}]")
    print(f"  Ours:     {obs_o:.2f}   95% CI [{ci_o[0]:.2f}, {ci_o[1]:.2f}]")
    print(f"  Δ Ours - Baseline = {obs_diff:+.2f}")

    # ------------------------------------------------------------
    # two-sided test
    # ------------------------------------------------------------
    if alternative == "two-sided":
        ci_diff = np.percentile(diff, [2.5, 97.5])

    
        if np.isclose(obs_diff, 0.0, atol=1e-12):
            p_value = 1.0
        elif obs_diff > 0:
            tail_count = np.sum(diff <= 0)
            p_value = min(1.0, 2.0 * (tail_count + 1) / (len(diff) + 1))
        else:
            tail_count = np.sum(diff >= 0)
            p_value = min(1.0, 2.0 * (tail_count + 1) / (len(diff) + 1))

        significant = (ci_diff[0] > 0) or (ci_diff[1] < 0)

        print(f"  Two-sided 95% CI of Δ: [{ci_diff[0]:+.2f}, {ci_diff[1]:+.2f}]")
        print(f"  two-sided p ≈ {p_value:.2f}")

        if significant:
            print("   Significant on both sides: The 95% CI of Δ does not include 0.")
        else:
            print("   Not significant on both sides: 95% CI for Δ includes 0.")

    # ------------------------------------------------------------
    # one-side test：Ours > Baseline
    # ------------------------------------------------------------
    elif alternative == "greater":
        lower_bound = np.percentile(diff, 5)

        p_value = (np.sum(diff <= 0) + 1) / (len(diff) + 1)

        significant = lower_bound > 0

        print("  One-sided hypothesis: Ours > Baseline")
        print(f"  One-sided 95% lower bound of Δ = {lower_bound:+.4f}")
        print(f"  one-sided p ≈ {p_value:.4f}")

        if significant:
            print("   One-sided significance: Ours is significantly greater than Baseline.")
        else:
            print("   One-sided insignificant.")

    # ------------------------------------------------------------
    # one-side test：Ours < Baseline
    # ------------------------------------------------------------
    elif alternative == "less":
        upper_bound = np.percentile(diff, 95)

        p_value = (np.sum(diff >= 0) + 1) / (len(diff) + 1)

        significant = upper_bound < 0

        print("  One-sided hypothesis: Ours < Baseline")
        print(f"  One-sided 95% upper bound of Δ = {upper_bound:+.4f}")
        print(f"  one-sided p ≈ {p_value:.4f}")

        if significant:
            print("   One-sided significance: Ours is significantly smaller than Baseline.")
        else:
            print("   One-sided insignificant.")

    else:
        raise ValueError("alternative can only be 'two-sided', 'greater' or 'less'。")

    print("-" * 75)


# ============================================================
# 8. Output
# ============================================================
print("\nPaired Bootstrap results")
print("=" * 75)

report_metric(
    name="AUC",
    obs_b=obs_auc_b,
    obs_o=obs_auc_o,
    boot_b=boot_auc_b,
    boot_o=boot_auc_o,
    alternative=ALT_AUC
)

report_metric(
    name="Recall (Class 1)",
    obs_b=obs_rec_b_c1,
    obs_o=obs_rec_o_c1,
    boot_b=boot_rec_b_c1,
    boot_o=boot_rec_o_c1,
    alternative=ALT_REC_C1
)

report_metric(
    name="Recall (Class 0)",
    obs_b=obs_rec_b_c0,
    obs_o=obs_rec_o_c0,
    boot_b=boot_rec_b_c0,
    boot_o=boot_rec_o_c0,
    alternative=ALT_REC_C0
)
